In [5]:
import torch.nn as nn
import torch
import torchvision
import torchvision.transforms.v2 as T
import numpy as np

toTensor = T.Compose([T.ToImage() , T.ToDtype(torch.float32 , scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets" , download=True , train=True , transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(root="datasets" , download=True , train=False , transform=toTensor)

In [6]:
torch.manual_seed(42)
train_data , valid_data = torch.utils.data.random_split(train_and_valid_data , [55000 , 5000])

In [7]:
from torch.utils.data import DataLoader

train_data_loader = DataLoader(train_data , batch_size=32 , shuffle=True )
test_data_loader = DataLoader(test_data , batch_size = 32)
valid_data_loader = DataLoader(valid_data , batch_size= 32)

In [8]:
#Classifier
class imageClassifier(nn.Module):
    def __init__(self, n_inputs , n_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs , 300),
            nn.ReLU(),
            nn.Linear(300, 200),
            nn.ReLU(),
            nn.Linear(200 , n_classes)
        )
    def forward(self , x):
        return self.model(x)
    
def train(model , criterion , optimizer , data_loader , n_epochs):
        model.to(device= "cuda")
        model.train()
        for epoch in range(n_epochs):
            total_loss = 0.0
            for x_batch, y_batch in data_loader:
                x_batch , y_batch = x_batch.to(device = "cuda") , y_batch.to(device = "cuda")
                y_pred = model(x_batch)
                loss = criterion(y_pred , y_batch)
                total_loss += loss.item()
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
            mean_loss = total_loss / len(data_loader)
            print(f"Epoch {epoch + 1} loss :{mean_loss}")

def evaluate(model , data_loader , metrics_fn  , aggregate_fn = lambda metric : torch.mean(metric)):
        model.to(device = "cuda")
        model.eval()
        metrics = []
        with torch.no_grad():
            for x_batch , y_batch in data_loader:
                x_batch , y_batch = x_batch.to(device = "cuda") , y_batch.to(device = "cuda")
                y_pred = model(x_batch)
                metric = metrics_fn(y_pred , y_batch)
                metrics.append(metric.detach())
            return aggregate_fn(torch.stack(metrics))

In [ ]:
model= imageClassifier(n_inputs= 28 * 28 , n_classes=10)
optimizer = torch.optim.SGD(model.parameters() , lr= 0.01)
criterion = nn.CrossEntropyLoss()
n_epochs = 1

train(model , criterion , optimizer , train_data_loader , n_epochs)

Epoch 1 loss :1.0943437282639648
Epoch 2 loss :0.5901380331711134
Epoch 3 loss :0.5067677744266043
Epoch 4 loss :0.46818474433132906
Epoch 5 loss :0.4440487916036842
Epoch 6 loss :0.42374940934726985
Epoch 7 loss :0.40934759124446013
Epoch 8 loss :0.3955622446431262
Epoch 9 loss :0.3835215515317856
Epoch 10 loss :0.37244827262602137
Epoch 11 loss :0.36300217503071736
Epoch 12 loss :0.3535266261051532
Epoch 13 loss :0.34547921128402
Epoch 14 loss :0.3377191983835166
Epoch 15 loss :0.32986510509907885
Epoch 16 loss :0.3239588910879577
Epoch 17 loss :0.3177219669769641
Epoch 18 loss :0.31074985696426516
Epoch 19 loss :0.3043790302212811
Epoch 20 loss :0.3001394409538962
Epoch 21 loss :0.29439637550586806
Epoch 22 loss :0.28887226217582934
Epoch 23 loss :0.2832454512260937
Epoch 24 loss :0.278986321407879
Epoch 25 loss :0.27552082763314073
Epoch 26 loss :0.269299809379398
Epoch 27 loss :0.26640774226850933
Epoch 28 loss :0.2612562751838501
Epoch 29 loss :0.25745564738614674
Epoch 30 loss :

In [10]:
import torchmetrics

accuracy = torchmetrics.Accuracy(task="multiclass" , num_classes=10).to(device="cuda")
evaluate(model , valid_data_loader , metrics_fn=accuracy)


tensor(0.8848, device='cuda:0')

In [14]:
#Hyper-parameter Tuning
import optuna

def objective(trial , train_data_loader , valid_data_loader): #Good practice
    learning_rate = trial.suggest_float("learning_rate" , 0.001 , 0.1 , log =True)
    model = imageClassifier(n_inputs= 28 * 28 , n_classes= 10).to(device="cuda")
    optimizer = torch.optim.SGD(model.parameters() , lr= learning_rate)
    train(model , criterion  , optimizer , train_data_loader , n_epochs)
    validation_accuracy = evaluate(model , valid_data_loader , metrics_fn= accuracy)
    return validation_accuracy



In [15]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study  = optuna.create_study(direction="maximize" , sampler=sampler)
objective_with_data = lambda trial : objective(trial , train_data_loader=train_data_loader , valid_data_loader=valid_data_loader) #argument Freeze
study.optimize(objective_with_data , n_trials=1)

[I 2026-01-24 21:35:24,842] A new study created in memory with name: no-name-7f34c73e-3e45-4817-b125-2efe6a8a8811


Epoch 1 loss :1.3778945510993523
Epoch 2 loss :0.6981558367705332
Epoch 3 loss :0.5844103432606652
Epoch 4 loss :0.5264601773165769
Epoch 5 loss :0.4921162272432365
Epoch 6 loss :0.4688731971922276
Epoch 7 loss :0.45187974082757876
Epoch 8 loss :0.43902132547459954
Epoch 9 loss :0.4274410966560548
Epoch 10 loss :0.4191348324548789
Epoch 11 loss :0.41013116319731546
Epoch 12 loss :0.402749014427802
Epoch 13 loss :0.3939169687221673
Epoch 14 loss :0.3876402670080888
Epoch 15 loss :0.38199449428522425
Epoch 16 loss :0.37502780414081854
Epoch 17 loss :0.3696595642015711
Epoch 18 loss :0.363741091894091
Epoch 19 loss :0.3589521932777033
Epoch 20 loss :0.35362035805416914
Epoch 21 loss :0.34827283005283766
Epoch 22 loss :0.34375350304498375
Epoch 23 loss :0.3394046420748321
Epoch 24 loss :0.33556367834426415
Epoch 25 loss :0.3308617593227534
Epoch 26 loss :0.3267973059286656
Epoch 27 loss :0.3219820188229482
Epoch 28 loss :0.31947927879534743
Epoch 29 loss :0.31453036571299703
Epoch 30 loss 

[I 2026-01-24 21:43:50,572] Trial 0 finished with value: 0.8730095624923706 and parameters: {'learning_rate': 0.005611516415334507}. Best is trial 0 with value: 0.8730095624923706.


In [16]:
torch.save(model , "my_fashion_mnist.pth")